# Tool Binding and Tool Calling

### 1. Tool Binding

Tool Binding is the process where you register tools with a LLM so that the LLM knows:

- What tools are available (via names)
- What each tool does (via description)
- What input and output format to use (via schema)

In [ ]:
# !pip install -q langchain-openai langchain-core requests

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

/Users/suryamgupta/Documents/Learning-LangChain/langvenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
# create tool

@tool
def multiply(a: int, b:int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b

In [3]:
multiply.invoke({"a": 3, "b": 4})

12

In [5]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given 2 numbers a and b this tool returns their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [6]:
llm = ChatOpenAI()

In [9]:
# tool binding
llm_with_tools = llm.bind_tools([multiply]) # can give more tools in the list

So now llm_with_tools is just an LLM with proper multiplication capabilities.

In [7]:
llm.invoke("What is 5*3")

AIMessage(content='5 * 3 equals 15.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 13, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFqFdTILoGd8jVbJeAfiPvOGf5t89', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cbb41-4c3b-79a2-bd61-1d205f8b8cbf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 8, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

ok works properly

In [10]:
llm_with_tools.invoke("What is 5*3")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 60, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFqFwhSLgRbxK3Gxmqrw4ozouu32E', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbb41-97da-7b51-9d86-b0c2a2382927-0', tool_calls=[{'name': 'multiply', 'args': {'a': 5, 'b': 3}, 'id': 'call_I2nL8eFccDHjCnAkIFJQkUws', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 17, 'total_tokens': 77, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

umm what

In [11]:
llm_with_tools.invoke("Who is Magnus Carlsen?")

AIMessage(content='Magnus Carlsen is a Norwegian chess grandmaster who is widely considered to be one of the greatest chess players of all time. He became a grandmaster at the age of 13 and has since dominated the world of chess, holding the highest Elo rating in the world for many years. Carlsen has won numerous prestigious chess tournaments and is the current World Chess Champion. He is known for his strategic and creative playing style, as well as his exceptional endgame skills.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 60, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFqGVRj20aUnvMc9Oou7YsKtSMcmt', 'service_tier': 'default',

In [15]:
llm_with_tools.invoke("What is 2*3")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 60, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFqHYp6TSa7t5pZ6CCdhVbQ1Yowey', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbb43-1e81-7671-97c0-3e045de69570-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'call_4NGqVm1Gm0UIRlhqJeZSCVXG', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 17, 'total_tokens': 77, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [13]:
llm_with_tools.invoke("Hello")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 55, 'total_tokens': 65, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFqHCsczKeNixFaiZ3hvDyyW3Nolg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cbb42-c624-7931-8951-07884b5bafa4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'output_tokens': 10, 'total_tokens': 65, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Great, so it doesn't do the only one thing that it was explicitly made to be perfect at

In [16]:
llm_with_tools.invoke("Do you have any tools inside of you?")

AIMessage(content='Yes, I have some tools available. What specific task or function are you looking to use a tool for?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 63, 'total_tokens': 86, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFqIlrHLqNTzQLu73xUhgihujJ21j', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cbb44-4049-7822-bde4-d3fc7caa2b7f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 23, 'total_tokens': 86, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [18]:
llm_with_tools.invoke("Do you have a multiplication tool inside of you?")

AIMessage(content='Yes, I have a multiplication tool inside of me. Would you like to use it for a specific calculation?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 64, 'total_tokens': 87, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFqJIJWgrxnWXBKTEui4KUEn3tNfn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cbb44-c3e6-78c3-a474-de51f86db41d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 23, 'total_tokens': 87, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [19]:
llm_with_tools.invoke("What is 5*3? Use multiplication tool.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 65, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFqJhuDH8MBpSzXuimHponGcGVWEL', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbb45-26e3-7d63-bb5a-765abb307517-0', tool_calls=[{'name': 'multiply', 'args': {'a': 5, 'b': 3}, 'id': 'call_7RfdSidZ1sl5QJexso8yxcho', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 65, 'output_tokens': 17, 'total_tokens': 82, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

#### Wait wait wait ok, turns out that apparently this was exactly what was supposed to happen, i just learned lol. Ok so here's the deal...

In [27]:
llm_with_tools.invoke("Hello how are you?")

AIMessage(content="I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with anything you need. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 59, 'total_tokens': 95, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFqZwH9ELpzza0gDbUtjrtvfrDpnx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cbb54-8035-77c3-a5d0-f79358bfaffe-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 59, 'output_tokens': 36, 'total_tokens': 95, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

so this works fine as usual

In [29]:
temp = llm_with_tools.invoke("What is 5*3")
temp

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 60, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFqbA50wRUtOACZ6MK2qTBUsWlDb9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbb55-aaec-7683-bc16-dff11a38d15c-0', tool_calls=[{'name': 'multiply', 'args': {'a': 5, 'b': 3}, 'id': 'call_T4TeGLHbA2DdfRnN87DRSNIM', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 17, 'total_tokens': 77, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

The content is indeed empty, but notice how there's a tool_calls arg here in the output, which is empty in the previous out

In [30]:
temp.tool_calls

[{'name': 'multiply',
  'args': {'a': 5, 'b': 3},
  'id': 'call_T4TeGLHbA2DdfRnN87DRSNIM',
  'type': 'tool_call'}]

In [31]:
temp.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 5, 'b': 3},
 'id': 'call_T4TeGLHbA2DdfRnN87DRSNIM',
 'type': 'tool_call'}

In [33]:
# see with another more textual and hence less obvious query
llm_with_tools.invoke("What is the product of 10 and 9?").tool_calls[0]

{'name': 'multiply',
 'args': {'a': 10, 'b': 9},
 'id': 'call_wwWQoh02ZGzSDN8GVVtoNoEj',
 'type': 'tool_call'}

#### So what's happening is

### 2. Tool Calling

Tool Calling is the process where LLM decides, during a conversation or task that it needs to use a specific tool (function), and generates a structured output with:

- Name of Tool
- Arguments to call it with

The LLM does not actually run the tool - it just suggests and advises the tool and the input arguments to send to the tool to get your answer, which it thinks might be helpful considering the query. The actual execution is handled by Langchain or us as programmers, which is also obvious and imp as we don't want LLM to directly use any tool it has and return the output, but we would want us to have that power to choose before ectual execution.


This is why tool_calls=[] in the "hello how are you" prompt, as the LLM doesn't think there's any need to call any tools for this query and returns the normal answer.


### Hence, 

### 3. Tool Execution

Tool Execution is the step where actual Python tool (function) is run using input args that the LLM suggested during tool calling. 


So it's like:

user: "what's 5*9"

LLM: "hey, just call "multiply" tool with input args 5 and 9"

user does multiply(5,9) and gets the answer!

In [34]:
# trying from scratch
result = llm_with_tools.invoke("What is 5*3")
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 5, 'b': 3},
 'id': 'call_ORi8pSreygjQFbZYGFMlaWHY',
 'type': 'tool_call'}

In [40]:
input_args = result.tool_calls[0]['args']
input_args

{'a': 5, 'b': 3}

In [41]:
multiply.invoke(input_args)

15

Great, but this is actually multiply tool working and giving a direct numeric answer, but ultimately, we want the LLM to give answer.

So to do that, instead of sending input_args = result.tool_calls[0]['args'] into multiply.invoke(), if we directly send result.tool_calls[0]...

In [42]:
multiply.invoke(result.tool_calls[0])

ToolMessage(content='15', name='multiply', tool_call_id='call_ORi8pSreygjQFbZYGFMlaWHY')

We get the same answer, but this time it is wrapped around nicely inside this ToolMessage(), which is a special message you get when a tool is executed from a tool call.

It's like one of those: AIMessage, HumanMessage, SystemMessage.

Now what you can do is, send this ToolMessage thing we just got back to LLM, essentially telling it, "hey, when i executed the tool with input args you suggested, i got this", and LLM sees this and returns answer in natual language.

#### hence, proper flow would be

In [45]:
# this is a human message
query = HumanMessage("What is 5*3?")
query

HumanMessage(content='What is 5*3?', additional_kwargs={}, response_metadata={})

In [46]:
# make a messages list which is going to act as a history of the conversation for LLM
messages = [query]
messages

[HumanMessage(content='What is 5*3?', additional_kwargs={}, response_metadata={})]

In [ ]:
# this will be AI message
result = llm_with_tools.invoke(messages)
result

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 61, 'total_tokens': 78, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFrBRul4LtguF90thoRolTNWvxT4w', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbb77-fcae-72f1-8d7e-23cab142941c-0', tool_calls=[{'name': 'multiply', 'args': {'a': 5, 'b': 3}, 'id': 'call_LKzhQ4IIRTEyl7enPjnN62I3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 17, 'total_tokens': 78, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [48]:
# this now has Human and AI message
messages.append(result)
messages

[HumanMessage(content='What is 5*3?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 61, 'total_tokens': 78, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFrBRul4LtguF90thoRolTNWvxT4w', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbb77-fcae-72f1-8d7e-23cab142941c-0', tool_calls=[{'name': 'multiply', 'args': {'a': 5, 'b': 3}, 'id': 'call_LKzhQ4IIRTEyl7enPjnN62I3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 17, 'total_tokens': 78, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outp

In [ ]:
# this is a Tool message
tool_result = multiply.invoke(result.tool_calls[0])
tool_result

ToolMessage(content='15', name='multiply', tool_call_id='call_LKzhQ4IIRTEyl7enPjnN62I3')

In [ ]:
# this now has Human, AI and Tool message, which acts as a history and full context of the conversation for LLM
messages.append(tool_result)
messages

[HumanMessage(content='What is 5*3?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 61, 'total_tokens': 78, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFrBRul4LtguF90thoRolTNWvxT4w', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbb77-fcae-72f1-8d7e-23cab142941c-0', tool_calls=[{'name': 'multiply', 'args': {'a': 5, 'b': 3}, 'id': 'call_LKzhQ4IIRTEyl7enPjnN62I3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 17, 'total_tokens': 78, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outp

In [52]:
final_result = llm_with_tools.invoke(messages)
final_result

AIMessage(content='The result of 5 * 3 is 15.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 86, 'total_tokens': 99, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFrENiotDoQjhZT4U8Of49kqDKy77', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cbb7a-c652-7ed0-9b0f-b9f0270fcabb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 86, 'output_tokens': 13, 'total_tokens': 99, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [53]:
final_result.content

'The result of 5 * 3 is 15.'

In [ ]:
# here it is altogether
from langchain_core.messages import HumanMessage, ToolMessage

query = "What is 5*3?"
ai_msg = llm_with_tools.invoke(query)

tool_call = ai_msg.tool_calls[0]
tool_result = multiply.invoke(tool_call["args"])

final = llm_with_tools.invoke([
    HumanMessage(content=query),
    ai_msg,
    ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"]),
])

print(final.content)  # should contain final natural-language answer

#### Now, most imp thing i wanted to try out, trying out a complicated calculation

In [57]:
query = "What is 5*3?"
llm.invoke(query).content

'5 * 3 = 15'

In [58]:
ai_msg = llm_with_tools.invoke(query)
tool_call = ai_msg.tool_calls[0]
tool_result = multiply.invoke(tool_call["args"])
final = llm_with_tools.invoke([
    HumanMessage(content=query),
    ai_msg,
    ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"]),
])
print(final.content)

5 multiplied by 3 is equal to 15.


In [59]:
query = "What is 345435*657675?"
llm.invoke(query).content

'The product of 345435 and 657675 is 227037581125'

In [60]:
ai_msg = llm_with_tools.invoke(query)
tool_call = ai_msg.tool_calls[0]
tool_result = multiply.invoke(tool_call["args"])
final = llm_with_tools.invoke([
    HumanMessage(content=query),
    ai_msg,
    ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"]),
])
print(final.content)

The product of 345435 and 657675 is 227,183,963,625.


This obviously is the correct and actual answer, not the previous one.

I also wondered and hence asked Claude:

i see, it does work. but i am still wondering, shouldnt there still be something in the content, like why is it just empty. it could have returned something but also have that tool_calls in the output as a suggestion, no? or if i am wrong just tell me and explain how things work here


Claude - 

You’re thinking about it correctly.
But in tool-calling mode, empty content is expected and often preferred.

Why it’s empty

When the model chooses a tool, that assistant turn is treated as a function-call turn, not a final-answer turn.

So it returns:

- tool_calls = “run this tool with these args”
- content = usually empty ("")

Could it return both?

Technically, some models/providers can return text + tool call together.

But many tool-call responses are intentionally content-empty to keep the protocol clean:
1. Decide tool + args
2. App executes tool
3. Tool result sent back
4. Model gives final human-readable answer

This avoids premature/incorrect guesses before seeing real tool output.

So yes: your result is normal, and your flow is working exactly as designed.